In [5]:

import json
import os
import joblib
import pandas as pd

MODEL = None
FEATURES = None


def init():
    global MODEL, FEATURES

    # Azure ML provides the registered model directory here
    model_dir = os.environ.get("AZUREML_MODEL_DIR")

    if not model_dir:
        raise RuntimeError("AZUREML_MODEL_DIR is not set")

    model_path = os.path.join(model_dir, "model.joblib")
    features_path = os.path.join(model_dir, "features.json")

    MODEL = joblib.load(model_path)

    with open(features_path, "r") as f:
        FEATURES = json.load(f)


def run(raw_data):
    try:
        data = json.loads(raw_data) if isinstance(raw_data, str) else raw_data

        transactions = data.get("transactions", [data])
        df = pd.DataFrame(transactions)

        missing = [f for f in FEATURES if f not in df.columns]
        if missing:
            return {
                "error": "Missing required features",
                "missing_features": missing,
            }

        df = df[FEATURES]

        probabilities = MODEL.predict_proba(df)[:, 1]
        predictions = (probabilities >= 0.5).astype(int)

        return {
            "predictions": [
                {
                    "is_fraud": int(prediction),
                    "fraud_probability": float(probability),
                }
                for prediction, probability in zip(
                    predictions, probabilities
                )
            ]
        }

    except Exception as exc:
        return {"error": str(exc)}


In [7]:

import numpy as np
import sys
import joblib

print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("Joblib:", joblib.__version__)


Python: /home/azureuser/cloudfiles/code/fraud-capstone-env/bin/python
NumPy: 1.23.5
Joblib: 1.6.0


In [10]:
import numpy as np
print(np.__version__)

1.23.5


In [11]:
import numpy as np
import sklearn
import imblearn
import joblib

print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("imbalanced-learn:", imblearn.__version__)
print("joblib:", joblib.__version__)

NumPy: 1.23.5
scikit-learn: 1.5.1
imbalanced-learn: 0.12.4
joblib: 1.6.0


In [13]:
import json
from pathlib import Path

PROJECT_DIR = Path.home() / "cloudfiles" / "code" / "capstone-project-1-fraud"

metadata_path = PROJECT_DIR / "metadata.json"

with open(metadata_path, "r") as f:
    metadata = json.load(f)

print(json.dumps(metadata, indent=2))

{
  "model_type": "Random Forest with SMOTE",
  "threshold": 0.5,
  "reported_test_precision": 0.9469,
  "reported_test_recall": 0.7815,
  "reported_test_f1": 0.85629,
  "reported_test_pr_auc": 0.88994
}


In [14]:

from pathlib import Path
import zipfile

model_path = (
    Path.home()
    / "cloudfiles"
    / "code"
    / "capstone-project-1-fraud"
    / "model.joblib"
)

print("Model file exists:", model_path.exists())
print("Model file size (MB):", round(model_path.stat().st_size / (1024 * 1024), 2))

with open(model_path, "rb") as f:
    header = f.read(5000)

print("\nReferences to NumPy core:", b"numpy._core" in header)
print("References to NumPy core multiarray:", b"numpy.core" in header)


Model file exists: True
Model file size (MB): 2.93

References to NumPy core: True
References to NumPy core multiarray: False


In [16]:

from pathlib import Path

home = Path.home()

for folder in [
    home / "cloudfiles" / "code" / "capstone-project-1-fraud",
    home / "cloudfiles" / "code",
]:
    print(f"\nFolder: {folder}")
    if folder.exists():
        for item in list(folder.iterdir())[:30]:
            print(" -", item.name)
    else:
        print("Folder not found")



Folder: /home/azureuser/cloudfiles/code/capstone-project-1-fraud
 - features.json
 - metadata.json
 - model.joblib
 - register_model.ipynb
 - score.ipynb

Folder: /home/azureuser/cloudfiles/code
 - capstone-project-1-fraud
 - fraud-capstone-env
 - Users


In [21]:

import numpy as np
import importlib.util

print("NumPy version:", np.__version__)
print("numpy._core exists:", importlib.util.find_spec("numpy._core") is not None)
print("numpy.core exists:", importlib.util.find_spec("numpy.core") is not None)


NumPy version: 1.23.5
numpy._core exists: False
numpy.core exists: True


In [2]:

from pathlib import Path

model_path = (
    Path.home()
    / "cloudfiles"
    / "code"
    / "capstone-project-1-fraud"
    / "model.joblib"
)

with open(model_path, "rb") as f:
    data = f.read()

for term in [b"numpy._core", b"numpy.core", b"numpy._core.multiarray"]:
    print(term.decode(), ":", data.count(term))


numpy._core : 6
numpy.core : 0
numpy._core.multiarray : 6


In [4]:

import sys
import numpy as np
import numpy.core

# Temporary compatibility aliases for older NumPy
# loading a model saved with newer NumPy module paths.
sys.modules["numpy._core"] = np.core
sys.modules["numpy._core.multiarray"] = np.core.multiarray

print("NumPy compatibility aliases configured.")
print("NumPy version:", np.__version__)


NumPy compatibility aliases configured.
NumPy version: 1.23.5


In [5]:

from pathlib import Path
import joblib
import json

PROJECT_DIR = Path.home() / "cloudfiles" / "code" / "capstone-project-1-fraud"

with open(PROJECT_DIR / "features.json", "r") as f:
    features = json.load(f)

model = joblib.load(PROJECT_DIR / "model.joblib")

print("Model loaded successfully!")
print("Model type:", type(model).__name__)
print("Features:", features)


Model loaded successfully!
Model type: Pipeline
Features: ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']


/home/azureuser/cloudfiles/code/fraud-capstone-env/lib/python3.10/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.7.2 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/azureuser/cloudfiles/code/fraud-capstone-env/lib/python3.10/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator NearestNeighbors from version 1.7.2 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/azureuser/cloudfiles/code/fraud-capstone-env/lib/python3.10/site-packages/sklearn/base.py:376: InconsistentVersionWarning: 

In [6]:

import pandas as pd

sample_transaction = {
    "step": 1,
    "type": "TRANSFER",
    "amount": 5000.0,
    "oldbalanceOrg": 5000.0,
    "newbalanceOrig": 0.0,
    "oldbalanceDest": 1000.0,
    "newbalanceDest": 6000.0,
}

transaction_df = pd.DataFrame([sample_transaction])[features]

probability = model.predict_proba(transaction_df)[0, 1]
prediction = int(probability >= 0.5)

print("Fraud probability:", round(float(probability), 4))
print("Prediction:", "FRAUD" if prediction else "NOT FRAUD")


Fraud probability: 0.1333
Prediction: NOT FRAUD


In [7]:
import sklearn
import numpy as np
import imblearn

print("scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("imbalanced-learn:", imblearn.__version__)

scikit-learn: 1.5.1
NumPy: 1.23.5
imbalanced-learn: 0.12.4


In [9]:

import json

with open(PROJECT_DIR / "metadata.json", "r") as f:
    metadata = json.load(f)

print("Model file:", PROJECT_DIR / "model.joblib")
print("Features:", features)
print("Threshold:", metadata.get("threshold", 0.5))
print("Model type:", metadata.get("model_type"))


Model file: /home/azureuser/cloudfiles/code/capstone-project-1-fraud/model.joblib
Features: ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
Threshold: 0.5
Model type: Random Forest with SMOTE


In [10]:
import joblib

print("Model file:", PROJECT_DIR / "model.joblib")
print("Features:", features)
print("Threshold:", metadata.get("threshold", 0.5))

Model file: /home/azureuser/cloudfiles/code/capstone-project-1-fraud/model.joblib
Features: ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
Threshold: 0.5


In [11]:
sample_transaction = {
    "step": 1,
    "type": "TRANSFER",
    "amount": 5000.0,
    "oldbalanceOrg": 5000.0,
    "newbalanceOrig": 0.0,
    "oldbalanceDest": 1000.0,
    "newbalanceDest": 6000.0
}

import pandas as pd

transaction_df = pd.DataFrame([sample_transaction])[features]

probability = model.predict_proba(transaction_df)[0, 1]
prediction = int(probability >= metadata.get("threshold", 0.5))

print("Fraud probability:", round(float(probability), 4))
print("Prediction:", "FRAUD" if prediction else "NOT FRAUD")

Fraud probability: 0.1333
Prediction: NOT FRAUD


In [2]:

import sys
import os
import subprocess
import importlib.metadata as metadata

print("=" * 60)
print("PYTHON ENVIRONMENT")
print("=" * 60)

print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("Virtual environment:", os.environ.get("VIRTUAL_ENV", "Not set"))
print("Current working directory:", os.getcwd())

print("\n" + "=" * 60)
print("IMPORTANT ML PACKAGE VERSIONS")
print("=" * 60)

packages = [
    "numpy",
    "scipy",
    "pandas",
    "scikit-learn",
    "imbalanced-learn",
    "joblib",
    "mlflow",
    "azure-ai-ml",
    "azure-identity",
    "fastapi",
    "uvicorn",
    "ipykernel"
]

for package in packages:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\n" + "=" * 60)
print("ALL INSTALLED PACKAGES")
print("=" * 60)

result = subprocess.run(
    [sys.executable, "-m", "pip", "list"],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("Warnings or errors:")
    print(result.stderr)


PYTHON ENVIRONMENT
Python version: 3.10.21 (main, Aug 27 2026, 14:42:07) [GCC 14.3.0]
Python executable: /anaconda/envs/azureml_py310_sdkv2/bin/python
Virtual environment: Not set
Current working directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/fraud-capstone-compute/code/capstone-project-1-fraud

IMPORTANT ML PACKAGE VERSIONS
numpy: 1.23.5
scipy: 1.15.3
pandas: 1.5.3
scikit-learn: 1.7.2
imbalanced-learn: NOT INSTALLED
joblib: 1.6.0
mlflow: 3.16.0
azure-ai-ml: 1.35.0
azure-identity: 1.17.0
fastapi: 0.141.1
uvicorn: 0.52.4
ipykernel: 7.3.0

ALL INSTALLED PACKAGES
Package                                Version
-------------------------------------- ------------
adal                                   1.2.7
aiohappyeyeballs                       2.7.1
aiohttp                                3.14.3
aiosignal                              1.4.0
alembic                                1.19.2
annotated-doc                          0.0.5
annotated-types                        0.8.0
anyio

In [3]:

import sys
import subprocess

print("Installing into:", sys.executable)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "imbalanced-learn==0.12.4"
])

print("Installation completed.")


Installing into: /anaconda/envs/azureml_py310_sdkv2/bin/python
  Using cached imbalanced_learn-0.12.4-py3-none-any.whl.metadata (8.3 kB)
Using cached imbalanced_learn-0.12.4-py3-none-any.whl (258 kB)
  Attempting uninstall: cloudpickle
    Found existing installation: cloudpickle 2.2.1
    Uninstalling cloudpickle-2.2.1:
      Successfully uninstalled cloudpickle-2.2.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [imbalanced-learn][imbalanced-learn]
Installation completed.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
azureml-dataprep 5.5.0 requires cloudpickle<3.0.0,>=1.1.0, but you have cloudpickle 3.1.2 which is incompatible.


In [5]:

import sys
import importlib.metadata as md

print("Python:", sys.executable)

for package in ["numpy", "scikit-learn", "imbalanced-learn", "cloudpickle"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: not installed")

print("\nChecking scikit-learn imports...")
from sklearn.pipeline import Pipeline
print("scikit-learn Pipeline import: OK")

print("\nChecking imbalanced-learn...")
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    print("imbalanced-learn Pipeline import: OK")
except Exception as e:
    print(type(e).__name__ + ":", e)


Python: /anaconda/envs/azureml_py310_sdkv2/bin/python
numpy: 1.23.5
scikit-learn: 1.7.2
imbalanced-learn: 0.12.4
cloudpickle: 3.1.2

Checking scikit-learn imports...
scikit-learn Pipeline import: OK

Checking imbalanced-learn...
imbalanced-learn Pipeline import: OK


In [7]:

import sys
import subprocess

print("Notebook Python:", sys.executable)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "imbalanced-learn==0.14.0"
])


Notebook Python: /anaconda/envs/azureml_py310_sdkv2/bin/python


  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5━━━━━ 0/2 [numpy]
  Attempting uninstall: imbalanced-learn━━━━━━━━ 0/2 [numpy]
    Found existing installation: imbalanced-learn 0.12.42 [numpy]
    Uninstalling imbalanced-learn-0.12.4:━━━ 0/2 [numpy]
      Successfully uninstalled imbalanced-learn-0.12.40/2 [numpy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [imbalanced-learn][imbalanced-learn]


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 3.16.0 requires mlflow-skinny==3.16.0, but you have mlflow-skinny 3.15.0 which is incompatible.


0

In [1]:

import numpy
import sklearn
import imblearn

print("NumPy:", numpy.__version__)
print("scikit-learn:", sklearn.__version__)
print("imbalanced-learn:", imblearn.__version__)
print("Imports successful")


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
source /home/azureuser/cloudfiles/code/fraud-inference-env/bin/activate 

In [ ]:
cd /home/azureuser/cloudfiles/code/capstone-project-1-fraud

In [ ]:
python -m uvicorn api.main:app --host 0.0.0.0 --port 8000

In [ ]:
curl http://127.0.0.1:8000/health